# EA-EMPD shocks: GC monetary events + ECB speeches → `Data/empd_shock.csv`

Builds the pooled policy-communication shock series from the EA-EMPD
(Altavilla, Gürkaynak, Kind, Laeven 2025, `EA-EMPD.xlsx`).

**Construction decisions**
- Events kept: `GC_ME` (Monetary Event window — the paper's MP shock concept) plus
  `EB` and `P` speeches. `GC_PR`/`GC_PC` are dropped: they are sub-windows of ME and
  would double count the meeting.
- Shock = `OIS_2Y` window surprise (bp), same units as the paper's EA-MPD shock.
  Events with missing `OIS_2Y` are dropped (counted).
- Day assignment (panel `delta_y` is close-to-close): events at hour ≥ 18 CET or on
  non-trading days go to the **next** trading day; everything else (incl. pre-open
  morning events) to the same trading day. Borderline: 17h speeches stay same-day.
- Multiple events per assigned day are **summed** (net daily policy news, incl.
  speech + meeting on the same day).
- Validation: GC_ME dates are checked against the calendar's `ecb_day` flags
  (the paper's 38 EA-MPD events).

In [1]:
import pandas as pd
import numpy as np

events = pd.read_excel("EA-EMPD.xlsx", sheet_name="EA-EMPD")
events["Date_time"] = pd.to_datetime(events["Date_time"])

cal = pd.read_csv("Data/release_shocks.csv", usecols=["date", "ecb_day"])
cal["date"] = pd.to_datetime(cal["date"])
cal = cal.sort_values("date").reset_index(drop=True)
caldates = cal["date"].values
print(f"calendar: {len(cal)} trading days, {cal['date'].min().date()} → {cal['date'].max().date()}, ecb_day: {int(cal['ecb_day'].sum())}")

calendar: 1238 trading days, 2021-01-04 → 2025-10-23, ecb_day: 38


In [2]:
ev = events[
    events["Event_type"].isin(["GC_ME", "EB", "P"])
    & (events["Date_time"] >= "2020-12-15")
].copy()
n_all = len(ev)
n_gc = int((ev["Event_type"] == "GC_ME").sum())
ev = ev.dropna(subset=["OIS_2Y"])
print(f"events in window: {n_all} ({n_gc} GC_ME), dropped for missing OIS_2Y: {n_all - len(ev)}")

# target calendar date: next day if evening or non-trading day
shift = (ev["Date_time"].dt.hour >= 18) | (ev["Non_regular_trading_day"] == 1)
target = ev["Date_time"].dt.normalize() + pd.to_timedelta(shift.astype(int), unit="D")
print(f"events shifted to next day (evening/non-trading): {int(shift.sum())}")

# snap to first trading day >= target
idx = np.searchsorted(caldates, target.values, side="left")
in_cal = idx < len(caldates)
print(f"events beyond calendar end (dropped): {int((~in_cal).sum())}")
ev, idx = ev[in_cal].copy(), idx[in_cal]
ev["business_date"] = caldates[idx]

events in window: 847 (39 GC_ME), dropped for missing OIS_2Y: 61
events shifted to next day (evening/non-trading): 89
events beyond calendar end (dropped): 3


In [3]:
# validation: GC_ME days vs the calendar's ecb_day flags (paper's EA-MPD events)
gc_days = set(pd.to_datetime(ev.loc[ev["Event_type"] == "GC_ME", "business_date"]).dt.date)
flag_days = set(cal.loc[cal["ecb_day"] == 1, "date"].dt.date)
print(f"GC_ME days in calendar window: {len(gc_days)}, ecb_day flags: {len(flag_days)}, matched: {len(gc_days & flag_days)}")
print("GC_ME days NOT flagged ecb_day:", sorted(gc_days - flag_days))
print("ecb_day flags with no GC_ME event:", sorted(flag_days - gc_days))

GC_ME days in calendar window: 38, ecb_day flags: 38, matched: 38
GC_ME days NOT flagged ecb_day: []
ecb_day flags with no GC_ME event: []


In [4]:
daily = (
    ev.groupby("business_date")
    .agg(empd_shock=("OIS_2Y", "sum"), n_empd_events=("OIS_2Y", "size"))
    .reset_index()
)

out = cal.rename(columns={"date": "business_date"}).merge(daily, on="business_date", how="left")
out["n_empd_events"] = out["n_empd_events"].fillna(0).astype(int)
out["empd_shock"] = out["empd_shock"].fillna(0.0)

evd = out[out["n_empd_events"] > 0]
print(f"trading days: {len(out)}, event days: {len(evd)}, multi-event days: {int((out['n_empd_events'] > 1).sum())}")
print("\nempd_shock on event days (bp):")
print(evd["empd_shock"].describe().round(3).to_string())
print("\n|shock| quantiles:", evd["empd_shock"].abs().quantile([0.5, 0.75, 0.9, 0.99]).round(2).tolist())
is_gc = out["ecb_day"] == 1
print(f"\nby day type: GC days n={int((evd['business_date'].isin(out.loc[is_gc,'business_date'])).sum())}, "
      f"sd={evd.loc[evd['business_date'].isin(out.loc[is_gc,'business_date']),'empd_shock'].std():.2f}bp | "
      f"speech-only days n={int((~evd['business_date'].isin(out.loc[is_gc,'business_date'])).sum())}, "
      f"sd={evd.loc[~evd['business_date'].isin(out.loc[is_gc,'business_date']),'empd_shock'].std():.2f}bp")

trading days: 1238, event days: 532, multi-event days: 180

empd_shock on event days (bp):
count    532.000
mean      -0.173
std        2.657
min      -19.238
25%       -0.749
50%       -0.004
75%        0.601
max       17.958

|shock| quantiles: [0.67, 1.62, 3.14, 10.44]

by day type: GC days n=38, sd=6.70bp | speech-only days n=494, sd=2.06bp


In [5]:
out["business_date"] = out["business_date"].dt.strftime("%Y-%m-%d")
out[["business_date", "empd_shock", "n_empd_events"]].to_csv("Data/empd_shock.csv", index=False)
print("written: Data/empd_shock.csv")

written: Data/empd_shock.csv
